<a href="https://colab.research.google.com/github/nic0les/6.s985_image_hallucination/blob/main/Copy_of_sdxl_orthographic_hallucination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDXL Orthographic Hallucination — Baseline & Mechanistic Experiments

This notebook implements:
- **Phase 0**: Controlled prompt suite + OCR evaluation pipeline (baseline CER/WER table)
- **Exp 1**: Text-encoding bottleneck analysis (tokenization, embedding separability, denoiser sensitivity)
- **Exp 2**: Cross-attention misbinding analysis (attention hooks, ROI statistics, scene complexity ablation)

**Runtime requirement**: GPU (T4 minimum, A100 recommended for faster generation). Enable via Runtime → Change runtime type → GPU.

---
## 0. Installation & Imports

In [ ]:
# Install dependencies
# Run this cell once; restart runtime after if prompted

# Uninstall HuggingFace libraries to ensure version consistency
!pip uninstall -y diffusers transformers accelerate peft huggingface_hub

# Reinstall HuggingFace libraries with compatible versions
# huggingface_hub==0.25.1 is chosen to satisfy peft's dependency
# accelerate==0.29.3 and peft==0.9.0 are compatible with transformers==4.41.0 and diffusers==0.27.2
!pip install -q huggingface_hub==0.25.1 transformers==4.41.0 accelerate==0.29.3 diffusers==0.27.2 peft==0.9.0

# Existing torch and torchvision installation (from previous successful steps)
!pip install torch==2.7.1+cu118 torchvision==0.22.0+cu118 --index-url https://download.pytorch.org/whl/cu118 --no-deps

# Other dependencies
!pip install -q xformers --index-url https://download.pytorch.org/whl/cu118

Found existing installation: diffusers 0.37.0
Uninstalling diffusers-0.37.0:
  Successfully uninstalled diffusers-0.37.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: huggingface_hub 1.7.1
Uninstalling huggingface_hub-1.7.1:
  Successfully uninstalled huggingface_hub-1.7.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.9 MB/s et

In [ ]:
!pip show diffusers huggingface_hub

Name: diffusers
Version: 0.27.2
Summary: State-of-the-art diffusion in PyTorch and JAX.
Home-page: https://github.com/huggingface/diffusers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/diffusers/graphs/contributors)
Author-email: patrick@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, importlib-metadata, numpy, Pillow, regex, requests, safetensors
Required-by: 
---
Name: huggingface-hub
Version: 0.25.1
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, packaging, pyyaml, requests, tqdm, typing-extensions
Required-by: accelerate, datasets, diffusers, gradio, gradio_cli

In [ ]:
import huggingface_hub
import diffusers
print(huggingface_hub.__version__)  # should be 0.23.4
print(diffusers.__version__)        # should be 0.27.2


0.25.1
0.27.2


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from pathlib import Path
from itertools import product
from collections import defaultdict
from scipy import stats
import json, os, re
from transformers import TrOCRProcessor, VisionEncoderDecoderModel


from Levenshtein import distance as levenshtein_distance

# Diffusers
from diffusers import StableDiffusionXLPipeline, DDIMScheduler
from diffusers.models.attention_processor import AttnProcessor2_0

# Transformers
from transformers import CLIPTokenizer, CLIPTextModel

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Using device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# Output directory
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'images').mkdir(exist_ok=True)
(OUTPUT_DIR / 'attention_maps').mkdir(exist_ok=True)
(OUTPUT_DIR / 'results').mkdir(exist_ok=True)
print('Output directories created.')

Output directories created.


---
## Phase 0: Prompt Suite & Baseline Pipeline

### 0.1 Define the Factorial Prompt Suite

In [ ]:
# ── Target strings across difficulty axes ──────────────────────────────────────
TARGET_STRINGS = {
    'short_common':    ['CAT', 'DOG', 'SUN', 'BIG', 'RED'],
    'medium_common':   ['MARKET', 'BRIDGE', 'GARDEN', 'WINDOW', 'YELLOW'],
    'long_common':     ['STRAWBERRY', 'THUNDERBOLT', 'WATERMELON'],
    'digits':          ['2847', '90312', '15503'],
    'mixed_alphanum':  ['H2O', 'R2D2', 'B52'],
}

# Flatten to a list of (string, category) tuples
ALL_TARGETS = [
    (s, cat)
    for cat, strings in TARGET_STRINGS.items()
    for s in strings
]

# ── Scene complexity ───────────────────────────────────────────────────────────
SCENE_TEMPLATES = {
    'blank':    'a plain white sign with the word "{target}" printed clearly on it',
    'simple':   'a wooden storefront sign that clearly says "{target}"',
    'cluttered':'a neon sign on a busy city street that reads "{target}"',
}

# ── Prompt format variants ─────────────────────────────────────────────────────
# Each is a function that takes (scene_desc, target) and returns a full prompt
PROMPT_FORMATS = {
    'quoted':    lambda scene, t: scene.replace(f'"{t}"', f'"{t}"'),
    'unquoted':  lambda scene, t: scene.replace(f'"{t}"', t),
    'letterspaced': lambda scene, t: scene.replace(f'"{t}"', ' '.join(list(t))),
    'verbatim':  lambda scene, t: scene.replace(f'"{t}"', t) + f', render the exact characters: {t}',
}

# ── Viewpoints ─────────────────────────────────────────────────────────────────
VIEWPOINTS = {
    'frontal':     '',  # no suffix
    'perspective': ', slight perspective angle, 3/4 view',
}

# ── Seeds ──────────────────────────────────────────────────────────────────────
SEEDS = [42, 123, 456, 789, 1337]

# ── Build prompt grid ──────────────────────────────────────────────────────────
def build_prompt_grid(targets=ALL_TARGETS,
                      scenes=SCENE_TEMPLATES,
                      formats=PROMPT_FORMATS,
                      viewpoints=VIEWPOINTS,
                      seeds=SEEDS):
    """
    Returns a list of dicts, each representing one generation job.
    For the full factorial design this can be large; we provide a
    `subset` flag to run a quick smoke-test first.
    """
    grid = []
    for (target, target_cat), (scene_name, scene_tmpl), (fmt_name, fmt_fn), (vp_name, vp_suffix) in product(
        targets, scenes.items(), formats.items(), viewpoints.items()
    ):
        scene_desc = scene_tmpl.format(target=target)
        prompt = fmt_fn(scene_desc, target) + vp_suffix
        for seed in seeds:
            grid.append({
                'target':      target,
                'target_cat':  target_cat,
                'scene':       scene_name,
                'format':      fmt_name,
                'viewpoint':   vp_name,
                'seed':        seed,
                'prompt':      prompt,
                'image_path':  None,  # filled in after generation
            })
    return grid


def build_smoke_test_grid():
    """Minimal grid for quick testing: 2 targets × 2 scenes × 1 format × 1 viewpoint × 2 seeds."""
    return build_prompt_grid(
        targets=[('MARKET', 'medium_common'), ('2847', 'digits')],
        scenes={k: SCENE_TEMPLATES[k] for k in ['blank', 'cluttered']},
        formats={k: PROMPT_FORMATS[k] for k in ['quoted']},
        viewpoints={k: VIEWPOINTS[k] for k in ['frontal']},
        seeds=[42, 123],
    )


# Choose which grid to run:
#   smoke_test_grid  → 8 images, fast, for debugging
#   full_grid        → ~2400 images, use for real experiments (consider subsampling)
SMOKE_TEST = True   # ← set to False for full run

prompt_grid = build_smoke_test_grid() if SMOKE_TEST else build_prompt_grid()
print(f'Prompt grid size: {len(prompt_grid)} generations')
pd.DataFrame(prompt_grid[:5])

Prompt grid size: 8 generations


,target,target_cat,scene,format,viewpoint,seed,prompt,image_path
0,MARKET,medium_common,blank,quoted,frontal,42,"a plain white sign with the word ""MARKET"" prin...",None
1,MARKET,medium_common,blank,quoted,frontal,123,"a plain white sign with the word ""MARKET"" prin...",None
2,MARKET,medium_common,cluttered,quoted,frontal,42,"a neon sign on a busy city street that reads ""...",None
3,MARKET,medium_common,cluttered,quoted,frontal,123,"a neon sign on a busy city street that reads ""...",None
4,2847,digits,blank,quoted,frontal,42,"a plain white sign with the word ""2847"" printe...",None


### 0.2 Load SDXL

In [ ]:
# Load SDXL base model
# fp16 + xformers keeps VRAM under 12 GB on T4
MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

# Enable memory-efficient attention if xformers is available
try:
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers enabled')
except Exception:
    print('xformers not available, using default attention')

# Fixed generation settings — never change these across experiments
GEN_KWARGS = dict(
    height=1024,
    width=1024,
    num_inference_steps=50,
    guidance_scale=7.5,
    negative_prompt='blurry, low quality, distorted',
)

print('SDXL loaded.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

model.fp16.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

...

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/5.14G [00:00<?, ?B/s]

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

xformers enabled
SDXL loaded.


### 0.3 Generate Images

In [ ]:
def generate_image(pipe, prompt, seed, save_path=None):
    """Generate a single image with a fixed seed and save it."""
    generator = torch.Generator(device=device).manual_seed(seed)
    with torch.inference_mode():
        result = pipe(prompt=prompt, generator=generator, **GEN_KWARGS)
    image = result.images[0]
    if save_path:
        image.save(save_path)
    return image


# Run generation over the prompt grid
# Skips already-generated images so you can resume interrupted runs
for i, entry in enumerate(prompt_grid):
    fname = f"{entry['target']}_{entry['scene']}_{entry['format']}_{entry['viewpoint']}_seed{entry['seed']}.png"
    save_path = OUTPUT_DIR / 'images' / fname
    entry['image_path'] = str(save_path)

    if save_path.exists():
        print(f'[{i+1}/{len(prompt_grid)}] Skipping (exists): {fname}')
        continue

    print(f'[{i+1}/{len(prompt_grid)}] Generating: {fname}')
    generate_image(pipe, entry['prompt'], entry['seed'], save_path=save_path)

print('\nAll generations complete.')

[1/8] Generating: MARKET_blank_quoted_frontal_seed42.png


  0%|          | 0/50 [00:00<?, ?it/s]

[2/8] Generating: MARKET_blank_quoted_frontal_seed123.png


  0%|          | 0/50 [00:00<?, ?it/s]

[3/8] Generating: MARKET_cluttered_quoted_frontal_seed42.png


...

[7/8] Generating: 2847_cluttered_quoted_frontal_seed42.png


  0%|          | 0/50 [00:00<?, ?it/s]

[8/8] Generating: 2847_cluttered_quoted_frontal_seed123.png


  0%|          | 0/50 [00:00<?, ?it/s]


All generations complete.


### 0.4 OCR Evaluation Pipeline

In [ ]:
trocr_processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
trocr_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed').to(device)
trocr_model.eval()
print('TrOCR ready.')

def run_ocr(image_path):
    """
    Returns list of (bbox, text, confidence) tuples
    """
    image = Image.open(image_path).convert('RGB')
    pixel_values = trocr_processor(image, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        generated_ids = trocr_model.generate(pixel_values)
    text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    if not text:
        return []
    # Return in same (bbox, text, confidence) format score_image expects
    return [([[0,0],[0,0],[0,0],[0,0]], text, 1.0)]


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TrOCR ready.


In [ ]:
# ── Metric functions ───────────────────────────────────────────────────────────

def ned(detected, target):
    """Normalized Edit Distance ∈ [0, 1]. 0 = perfect match."""
    if len(detected) == 0 and len(target) == 0:
        return 0.0
    return levenshtein_distance(detected.upper(), target.upper()) / max(len(detected), len(target))


def cer(detected, target):
    """Character Error Rate = edit_distance / len(target). Can exceed 1."""
    if len(target) == 0:
        return 0.0 if len(detected) == 0 else 1.0
    return levenshtein_distance(detected.upper(), target.upper()) / len(target)


def wer(detected, target):
    """Word Error Rate at word level."""
    d_words = detected.upper().split()
    t_words = target.upper().split()
    if len(t_words) == 0:
        return 0.0 if len(d_words) == 0 else 1.0
    return levenshtein_distance(d_words, t_words) / len(t_words)


def classify_failure(detected, target, ned_score):
    """Classify the type of orthographic hallucination."""
    if detected == '':
        return 'no_text'
    if ned_score == 0.0:
        return 'success'
    if ned_score <= 0.1:
        return 'near_miss'
    # Check for visually similar glyph substitutions
    glyph_subs = {'O': '0', '0': 'O', 'I': '1', '1': 'I', 'E': '3', 'S': '5', 'B': '8'}
    cleaned = detected.upper()
    for orig, sub in glyph_subs.items():
        cleaned = cleaned.replace(sub, orig)
    if ned(cleaned, target) <= 0.1:
        return 'glyph_substitution'
    if ned_score > 0.8:
        return 'garbled'
    return 'partial'



# ── Per-image OCR scoring ──────────────────────────────────────────────────────

def score_image(image_path, target, confidence_threshold=0.3):
    """
    Run OCR on an image and return a dict of metrics.
    Handles no-detection and multi-detection cases.
    """
    results = run_ocr(image_path)

    if not results:
        return {
            'detected_string': '',
            'ocr_confidence':  0.0,
            'ned':             1.0,
            'cer':             1.0,
            'wer':             1.0,
            'success_strict':  False,
            'success_lenient': False,
            'failure_type':    'no_text',
            'n_detections':    0,
            'spurious_text':   False,
        }

    # Sort detections top-to-bottom, left-to-right
    results_sorted = sorted(results, key=lambda r: (r[0][0][1], r[0][0][0]))

    # Among all detections, find the one with best NED against target
    best_ned = 1.0
    best_str = ''
    best_conf = 0.0

    # Also try concatenating all detections
    concat_str = ''.join(r[1] for r in results_sorted if r[2] >= confidence_threshold)

    candidates = [(r[1], r[2]) for r in results_sorted] + [(concat_str, np.mean([r[2] for r in results_sorted]))]

    for cand_str, cand_conf in candidates:
        n = ned(cand_str, target)
        if n < best_ned:
            best_ned = n
            best_str = cand_str
            best_conf = cand_conf

    # Spurious text: any high-confidence detection that doesn't match target
    spurious = any(
        r[2] >= confidence_threshold and ned(r[1], target) > 0.5
        for r in results_sorted
    )

    ned_score = best_ned
    return {
        'detected_string': best_str,
        'ocr_confidence':  float(best_conf),
        'ned':             float(ned_score),
        'cer':             float(cer(best_str, target)),
        'wer':             float(wer(best_str, target)),
        'success_strict':  best_str.upper().strip() == target.upper().strip(),
        'success_lenient': ned_score <= 0.1,
        'failure_type':    classify_failure(best_str, target, ned_score),
        'n_detections':    len(results),
        'spurious_text':   spurious,
    }


print('Metric functions defined.')

Metric functions defined.


In [ ]:
# Run OCR evaluation over all generated images
results_records = []

for i, entry in enumerate(prompt_grid):
    if entry['image_path'] is None or not Path(entry['image_path']).exists():
        print(f'[{i+1}] Missing image, skipping: {entry}')
        continue

    scores = score_image(entry['image_path'], entry['target'])
    record = {**entry, **scores}
    results_records.append(record)

    print(f"[{i+1}/{len(prompt_grid)}] {entry['target']:15s} "
          f"scene={entry['scene']:10s} fmt={entry['format']:12s} "
          f"detected='{scores['detected_string']:15s}' "
          f"CER={scores['cer']:.2f} type={scores['failure_type']}")

# Save results
results_df = pd.DataFrame(results_records)
results_df.to_csv(OUTPUT_DIR / 'results' / 'baseline_scores.csv', index=False)
print(f'\nResults saved. Shape: {results_df.shape}')

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


[1/8] MARKET          scene=blank      fmt=quoted       detected='MAR:           ' CER=0.50 type=partial
[2/8] MARKET          scene=blank      fmt=quoted       detected='CASHIER        ' CER=0.83 type=partial
[3/8] MARKET          scene=cluttered  fmt=quoted       detected='ITEM           ' CER=0.83 type=garbled
[4/8] MARKET          scene=cluttered  fmt=quoted       detected='ALL            ' CER=0.83 type=garbled
[5/8] 2847            scene=blank      fmt=quoted       detected='2%             ' CER=0.75 type=partial
[6/8] 2847            scene=blank      fmt=quoted       detected='               ' CER=1.00 type=no_text
[7/8] 2847            scene=cluttered  fmt=quoted       detected='               ' CER=1.00 type=no_text
[8/8] 2847            scene=cluttered  fmt=quoted       detected='2897-287       ' CER=1.25 type=partial

Results saved. Shape: (8, 18)


### 0.5 Baseline Visualization

In [ ]:
def bootstrap_ci(values, n_boot=1000, ci=95):
    """Return (mean, lower_ci, upper_ci) via bootstrapping."""
    values = np.array(values)
    boot_means = [np.mean(np.random.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot_means, (100 - ci) / 2)
    hi = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return np.mean(values), lo, hi


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Baseline CER by Experimental Axis', fontsize=14, fontweight='bold')

# ── Plot 1: CER by target category ────────────────────────────────────────────
ax = axes[0]
cat_order = ['short_common', 'medium_common', 'long_common', 'digits', 'mixed_alphanum']
cat_stats = []
for cat in cat_order:
    subset = results_df[results_df['target_cat'] == cat]['cer'].values
    if len(subset) > 0:
        mean, lo, hi = bootstrap_ci(subset)
        cat_stats.append({'cat': cat.replace('_', '\n'), 'mean': mean, 'lo': lo, 'hi': hi})

cat_df = pd.DataFrame(cat_stats)
bars = ax.bar(cat_df['cat'], cat_df['mean'], color='steelblue', alpha=0.8)
ax.errorbar(range(len(cat_df)), cat_df['mean'],
            yerr=[cat_df['mean'] - cat_df['lo'], cat_df['hi'] - cat_df['mean']],
            fmt='none', color='black', capsize=4)
ax.set_ylabel('Mean CER')
ax.set_title('By String Category')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='CER=0.5')
ax.legend(fontsize=8)

# ── Plot 2: CER by scene complexity ───────────────────────────────────────────
ax = axes[1]
scene_stats = []
for scene in ['blank', 'simple', 'cluttered']:
    subset = results_df[results_df['scene'] == scene]['cer'].values
    if len(subset) > 0:
        mean, lo, hi = bootstrap_ci(subset)
        scene_stats.append({'scene': scene, 'mean': mean, 'lo': lo, 'hi': hi})

scene_df = pd.DataFrame(scene_stats)
colors = ['#2ecc71', '#f39c12', '#e74c3c']
ax.bar(scene_df['scene'], scene_df['mean'], color=colors, alpha=0.8)
ax.errorbar(range(len(scene_df)), scene_df['mean'],
            yerr=[scene_df['mean'] - scene_df['lo'], scene_df['hi'] - scene_df['mean']],
            fmt='none', color='black', capsize=4)
ax.set_ylabel('Mean CER')
ax.set_title('By Scene Complexity')
ax.set_ylim(0, 1.05)

# ── Plot 3: CER by prompt format ──────────────────────────────────────────────
ax = axes[2]
fmt_stats = []
for fmt in ['quoted', 'unquoted', 'letterspaced', 'verbatim']:
    subset = results_df[results_df['format'] == fmt]['cer'].values
    if len(subset) > 0:
        mean, lo, hi = bootstrap_ci(subset)
        fmt_stats.append({'fmt': fmt, 'mean': mean, 'lo': lo, 'hi': hi})

fmt_df = pd.DataFrame(fmt_stats)
ax.bar(fmt_df['fmt'], fmt_df['mean'], color='mediumpurple', alpha=0.8)
ax.errorbar(range(len(fmt_df)), fmt_df['mean'],
            yerr=[fmt_df['mean'] - fmt_df['lo'], fmt_df['hi'] - fmt_df['mean']],
            fmt='none', color='black', capsize=4)
ax.set_ylabel('Mean CER')
ax.set_title('By Prompt Format')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results' / 'baseline_cer_by_axis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Baseline plots saved.')

In [ ]:
# Failure type distribution heatmap: rows=target_cat, cols=failure_type
failure_pivot = (
    results_df.groupby(['target_cat', 'failure_type'])
    .size()
    .unstack(fill_value=0)
)
# Normalize to proportions per row
failure_pivot_pct = failure_pivot.div(failure_pivot.sum(axis=1), axis=0)

plt.figure(figsize=(10, 5))
sns.heatmap(failure_pivot_pct, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Proportion'})
plt.title('Failure Type Distribution by Target Category')
plt.ylabel('Target Category')
plt.xlabel('Failure Type')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'results' / 'failure_type_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()